# Assignment 8

First we need to import the different libraries needed in the project.

In [ ]:
import os
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import json
import cv2
import matplotlib.pyplot as plt

print("Successfully imported all dependencies!")

Now we define the function to be used.

In [ ]:
def extract_joint_nodes(video_path: str, model_path: str='../data/pose_landmarker_full.task'):
	base_options = python.BaseOptions(model_path)
	options = vision.PoseLandmarkerOptions(
		base_options=base_options,
		running_mode=vision.RunningMode.IMAGE,
		num_poses=1,
		min_pose_detection_confidence=0.5,
		min_tracking_confidence=0.5,
		output_segmentation_masks=False
	)

	cap = cv2.VideoCapture(video_path)
	frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
	frame_num = 0

	all_landmarks_data = {}

	with vision.PoseLandmarker.create_from_options(options) as landmarker:
		while cap.isOpened():

			success, frame = cap.read()
			if success:
				rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
				mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

				result = landmarker.detect(mp_image)

				frame_data = {
					"landmarks": []
				}
				if result.pose_landmarks:
					for detection in result.pose_landmarks:
						frame_data["landmarks"] = ([{"id":idx, "x":detection[idx].x, "y":detection[idx].y} for idx in range(len(detection))])
				
				all_landmarks_data[frame_num] = frame_data

				frame_num += 1
			
			else:
				break

	cap.release()

	return all_landmarks_data


And now we demonstrate that the function works as intended!

In [ ]:
def display_frame(image, node_data):

	height, width, _ = image.shape
	plt.figure(figsize=[8,12])
	plt.imshow(image)
	
	POSE_CONNECTIONS = [
        # Face
        (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8), (9, 10),
        # Arms
        (11, 13), (13, 15), (15, 21), (15, 17), (17, 19), (15, 19), (12, 14), (14, 16), (16, 22), (16, 18), (18, 20), (16, 20),
        # Torso
        (11, 12), (11, 23), (12, 24), (23, 24),
        # Legs
        (23, 25), (25, 27), (27, 29), (29, 31), (27, 31), (24, 26), (26, 28), (28, 30), (30, 32), (28, 32)
    ]

	nodes = node_data["landmarks"]

	for connection in POSE_CONNECTIONS:
		start_idx, end_idx = connection
		if start_idx < len(nodes) and end_idx < len(nodes):
			start_point = nodes[start_idx]
			end_point = nodes[end_idx]
			plt.plot([start_point["x"] * width, end_point["x"] * width], [start_point["y"] * height, end_point["y"] * height], "r-")
	
	plt.axis("off")
	plt.show()

video_path = "../../EnisProject/data/test_squat.mp4"
model_path = '../data/pose_landmarker_full.task'

nodes = extract_joint_nodes(video_path)

chosen_frame = 60

cap = cv2.VideoCapture(video_path)

if chosen_frame > len(nodes):
	chosen_frame = len(nodes) - 1

for _ in range(chosen_frame):
	success, image = cap.read()

if success:
	display_frame(cv2.cvtColor(image, cv2.COLOR_BGR2RGB), nodes[chosen_frame])
cap.release()

Lastly we need to save this node data.

In [ ]:
def save_to_json(video_path: str, save_path: str, node_data: dict):
	with open(save_path, "w") as f:
		json.dump({
			"video_name": os.path.basename(video_path),
			"total_frames": len(node_data),
			"frames": node_data
		}, f, indent=2)
	return True
    
output_json_path = "../data/test_squat_landmarks.json"

if save_to_json(video_path, output_json_path, nodes):
	print("Successfully saved node data!")